# 🥉 BRONZE LAYER — Ingest Data Mentah ke Parquet

**Tujuan:** Membaca semua file CSV mentah dari World Bank Gender Statistics dan menyimpannya ke format Parquet **tanpa transformasi apapun**.

| Input | Output |
|-------|--------|
| `/data/raw/Gender_StatsCSV.csv` (101 MB) | `/data/bronze/main_data.parquet` |
| `/data/raw/Gender_StatsCountry.csv` (153 KB) | `/data/bronze/country_meta.parquet` |
| `/data/raw/Gender_StatsSeries.csv` (3.1 MB) | `/data/bronze/series_meta.parquet` |

In [12]:
from pyspark.sql import SparkSession  # type: ignore
from pyspark.sql.types import StructType, StructField, StringType  # type: ignore

# 1. Inisialisasi SparkSession (local mode — lebih cepat untuk dataset <1GB)
spark = SparkSession.builder \
    .appName("SDG5-Gender-Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .config("mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Mode: local[*] (optimal untuk dataset 100MB)")

Spark Version: 3.5.0
Mode: local[*] (optimal untuk dataset 100MB)


In [13]:
# 2. Definisi Schema Eksplisit (hindari inferSchema yang lambat pada file besar)
year_cols = [StructField(str(y), StringType(), True) for y in range(1960, 2026)]
main_schema = StructType([
    StructField("Country Name", StringType(), True),
    StructField("Country Code", StringType(), True),
    StructField("Indicator Name", StringType(), True),
    StructField("Indicator Code", StringType(), True)
] + year_cols)

print(f"Schema didefinisikan: 4 kolom identifier + {len(year_cols)} kolom tahun (1960-2025)")
print(f"Total kolom: {4 + len(year_cols)}")

Schema didefinisikan: 4 kolom identifier + 66 kolom tahun (1960-2025)
Total kolom: 70


In [14]:
# 3. Load Data CSV
print("Membaca data mentah CSV...")

# File utama (368.880 baris, 70 kolom)
df_main = spark.read.csv(
    "/data/raw/Gender_StatsCSV.csv",
    schema=main_schema,
    header=True,
    quote='"',
    escape='"'
)

# Metadata negara (265 baris)
df_country = spark.read.csv(
    "/data/raw/Gender_StatsCountry.csv",
    header=True,
    inferSchema=False
)

# Metadata indikator (1.392 baris)
df_series = spark.read.csv(
    "/data/raw/Gender_StatsSeries.csv",
    header=True,
    inferSchema=False
)

print("Data berhasil dibaca.")

Membaca data mentah CSV...
Data berhasil dibaca.


In [15]:
# 4. Simpan ke format Parquet (Bronze — tanpa ubahan apapun)
print("Menyimpan ke Parquet (Bronze)... Mohon tunggu, jangan tekan Run lagi.")

df_main.coalesce(1).write.mode("overwrite").parquet("/data/bronze/main_data.parquet")
df_country.coalesce(1).write.mode("overwrite").parquet("/data/bronze/country_meta.parquet")
df_series.coalesce(1).write.mode("overwrite").parquet("/data/bronze/series_meta.parquet")

print("✅ Parquet berhasil ditulis.")

Menyimpan ke Parquet (Bronze)... Mohon tunggu, jangan tekan Run lagi.
✅ Parquet berhasil ditulis.


In [16]:
# 5. Verifikasi Akhir (baca dari parquet yang sudah ditulis, bukan dari CSV lagi)
print("=" * 50)
print("BRONZE VERIFICATION")
print("=" * 50)

df_main_verify = spark.read.parquet("/data/bronze/main_data.parquet")
df_country_verify = spark.read.parquet("/data/bronze/country_meta.parquet")
df_series_verify = spark.read.parquet("/data/bronze/series_meta.parquet")

print(f"Main data rows : {df_main_verify.count():,}")
print(f"Main data cols : {len(df_main_verify.columns)}")
print(f"Country meta   : {df_country_verify.count()} rows, {len(df_country_verify.columns)} cols")
print(f"Series meta    : {df_series_verify.count()} rows, {len(df_series_verify.columns)} cols")
print("=" * 50)
print("✅ PROSES BRONZE LAYER SELESAI!")

BRONZE VERIFICATION
Main data rows : 368,880
Main data cols : 70
Country meta   : 271 rows, 31 cols
Series meta    : 3318 rows, 20 cols
✅ PROSES BRONZE LAYER SELESAI!
